# Stock Price Prediction & Algorithmic Trading

## Problem Overview

This notebook implements an algorithmic trading system that predicts stock price movements and executes trades to maximize profit.

**Game Rules:**
- Start with $100
- Each day: receive current + previous 4 days of stock prices
- Decide to BUY or SELL stocks
- Money from sales is only available the NEXT day
- Goal: Maximize score = 5 × ln(final_money)

**Strategy:**
- Use XGBoost for price movement prediction
- Technical indicators: moving averages, momentum, volatility
- Risk management: position sizing, diversification
- State persistence across trading days

In [ ]:
# Cell 1: Setup and Imports
import os
import json
import pickle
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from dataclasses import dataclass, field
from datetime import datetime

# ML Libraries
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Snowflake
from snowflake.snowpark.context import get_active_session

# Get active Snowpark session
try:
    session = get_active_session()
    print("✓ Connected to Snowflake")
except:
    print("⚠ No active Snowpark session - running in local mode")
    session = None

# Configuration
STATE_DIR = "/tmp/stock_trading_state"
os.makedirs(STATE_DIR, exist_ok=True)

print(f"State directory: {STATE_DIR}")
print("Setup complete!")

## Data Structures for Portfolio Management

In [ ]:
# Cell 2: Portfolio and Trading Data Structures

@dataclass
class StockHolding:
    """Represents holdings in a single stock"""
    name: str
    shares: int
    price_history: List[float] = field(default_factory=list)
    
    def current_price(self) -> float:
        """Get most recent price"""
        return self.price_history[-1] if self.price_history else 0.0
    
    def current_value(self) -> float:
        """Calculate current value of holdings"""
        return self.shares * self.current_price()


@dataclass
class Portfolio:
    """Manages portfolio state and transactions"""
    cash: float = 100.0
    pending_cash: float = 0.0  # Cash from sales, available tomorrow
    holdings: Dict[str, StockHolding] = field(default_factory=dict)
    transaction_history: List[Dict] = field(default_factory=list)
    day: int = 0
    
    def total_value(self) -> float:
        """Calculate total portfolio value"""
        stock_value = sum(h.current_value() for h in self.holdings.values())
        return self.cash + stock_value
    
    def can_buy(self, stock_name: str, shares: int, price: float) -> bool:
        """Check if we have enough cash to buy"""
        cost = shares * price
        return cost <= self.cash
    
    def can_sell(self, stock_name: str, shares: int) -> bool:
        """Check if we have enough shares to sell"""
        if stock_name not in self.holdings:
            return False
        return self.holdings[stock_name].shares >= shares
    
    def execute_buy(self, stock_name: str, shares: int, price: float):
        """Execute a buy transaction"""
        cost = shares * price
        if not self.can_buy(stock_name, shares, price):
            raise ValueError(f"Insufficient cash to buy {shares} shares of {stock_name}")
        
        self.cash -= cost
        
        if stock_name not in self.holdings:
            self.holdings[stock_name] = StockHolding(stock_name, 0)
        
        self.holdings[stock_name].shares += shares
        
        self.transaction_history.append({
            'day': self.day,
            'action': 'BUY',
            'stock': stock_name,
            'shares': shares,
            'price': price,
            'value': cost
        })
    
    def execute_sell(self, stock_name: str, shares: int, price: float):
        """Execute a sell transaction"""
        if not self.can_sell(stock_name, shares):
            raise ValueError(f"Insufficient shares to sell {shares} of {stock_name}")
        
        proceeds = shares * price
        self.pending_cash += proceeds  # Available tomorrow
        self.holdings[stock_name].shares -= shares
        
        self.transaction_history.append({
            'day': self.day,
            'action': 'SELL',
            'stock': stock_name,
            'shares': shares,
            'price': price,
            'value': proceeds
        })
    
    def start_new_day(self):
        """Move to next trading day - pending cash becomes available"""
        self.cash += self.pending_cash
        self.pending_cash = 0.0
        self.day += 1
    
    def get_score(self) -> float:
        """Calculate current score"""
        total = self.total_value()
        return 5.0 * np.log(total) if total > 0 else 0.0


print("Portfolio data structures defined")

## Feature Engineering for Stock Prediction

In [ ]:
# Cell 3: Feature Engineering Functions

def calculate_technical_features(prices: List[float]) -> Dict[str, float]:
    """
    Calculate technical indicators from price history.
    
    Args:
        prices: List of prices (oldest to newest, length >= 5)
    
    Returns:
        Dictionary of features
    """
    if len(prices) < 5:
        raise ValueError("Need at least 5 days of prices")
    
    prices = np.array(prices)
    current_price = prices[-1]
    
    features = {}
    
    # Raw prices (last 5 days)
    for i in range(5):
        features[f'price_t_minus_{4-i}'] = prices[-(5-i)]
    
    # Returns (day-over-day percentage changes)
    returns = np.diff(prices) / prices[:-1]
    for i in range(min(4, len(returns))):
        features[f'return_t_minus_{3-i}'] = returns[-(4-i)]
    
    # Moving averages
    if len(prices) >= 3:
        features['ma_3'] = np.mean(prices[-3:])
    if len(prices) >= 5:
        features['ma_5'] = np.mean(prices[-5:])
    
    # Price momentum (current vs X days ago)
    features['momentum_3d'] = (current_price / prices[-3] - 1.0) if len(prices) >= 3 else 0.0
    features['momentum_5d'] = (current_price / prices[-5] - 1.0) if len(prices) >= 5 else 0.0
    
    # Volatility (standard deviation of returns)
    if len(returns) >= 4:
        features['volatility'] = np.std(returns[-4:])
    else:
        features['volatility'] = 0.0
    
    # Trend indicators
    features['price_vs_ma5'] = (current_price / features.get('ma_5', current_price) - 1.0)
    
    # RSI-like indicator (simplified)
    if len(returns) >= 4:
        gains = returns[-4:][returns[-4:] > 0].sum()
        losses = abs(returns[-4:][returns[-4:] < 0].sum())
        features['rsi'] = gains / (gains + losses + 1e-10)
    else:
        features['rsi'] = 0.5
    
    return features


def create_training_dataset(price_histories: Dict[str, List[float]]) -> pd.DataFrame:
    """
    Create training dataset from historical prices.
    
    For each stock and each time window, extract features and label
    (next day's return).
    """
    training_data = []
    
    for stock_name, prices in price_histories.items():
        if len(prices) < 6:  # Need at least 6 days (5 for features, 1 for label)
            continue
        
        # Create sliding windows
        for i in range(len(prices) - 5):
            window = prices[i:i+5]
            next_price = prices[i+5]
            
            # Calculate features
            features = calculate_technical_features(window)
            features['stock'] = stock_name
            
            # Calculate label (next day's return)
            current_price = window[-1]
            next_return = (next_price / current_price - 1.0)
            features['target_return'] = next_return
            features['target_direction'] = 1 if next_return > 0 else 0
            
            training_data.append(features)
    
    return pd.DataFrame(training_data)


# Test feature calculation
test_prices = [4.54, 5.53, 6.56, 5.54, 7.60]
test_features = calculate_technical_features(test_prices)
print("Sample features:")
for k, v in list(test_features.items())[:5]:
    print(f"  {k}: {v:.4f}")
print(f"  ... ({len(test_features)} features total)")

## ML Model: XGBoost Price Predictor

In [ ]:
# Cell 4: ML Model Training

class StockPredictor:
    """
    XGBoost-based stock price movement predictor.
    Predicts next-day returns based on technical features.
    """
    
    def __init__(self):
        self.model = None
        self.feature_names = None
        self.is_trained = False
    
    def train(self, X: pd.DataFrame, y: np.ndarray, validation_split: float = 0.2):
        """
        Train the prediction model.
        
        Args:
            X: Feature dataframe
            y: Target values (returns)
            validation_split: Fraction of data for validation
        """
        # Store feature names
        self.feature_names = X.columns.tolist()
        
        # Split data
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=validation_split, random_state=42
        )
        
        # Train XGBoost regressor
        self.model = xgb.XGBRegressor(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective='reg:squarederror'
        )
        
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        self.is_trained = True
        
        # Evaluate
        train_pred = self.model.predict(X_train)
        val_pred = self.model.predict(X_val)
        
        train_rmse = np.sqrt(np.mean((train_pred - y_train) ** 2))
        val_rmse = np.sqrt(np.mean((val_pred - y_val) ** 2))
        
        # Directional accuracy
        train_direction_acc = np.mean((train_pred > 0) == (y_train > 0))
        val_direction_acc = np.mean((val_pred > 0) == (y_val > 0))
        
        print(f"Training RMSE: {train_rmse:.4f}")
        print(f"Validation RMSE: {val_rmse:.4f}")
        print(f"Training Direction Accuracy: {train_direction_acc:.2%}")
        print(f"Validation Direction Accuracy: {val_direction_acc:.2%}")
    
    def predict(self, features: Dict[str, float]) -> float:
        """
        Predict next-day return.
        
        Args:
            features: Dictionary of technical features
        
        Returns:
            Predicted return (e.g., 0.05 for 5% gain)
        """
        if not self.is_trained:
            # Default to simple momentum if not trained
            return features.get('momentum_3d', 0.0)
        
        # Create feature vector in correct order
        X = pd.DataFrame([features])[self.feature_names]
        return self.model.predict(X)[0]
    
    def save(self, filepath: str):
        """Save model to file"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'model': self.model,
                'feature_names': self.feature_names,
                'is_trained': self.is_trained
            }, f)
    
    def load(self, filepath: str):
        """Load model from file"""
        if os.path.exists(filepath):
            with open(filepath, 'rb') as f:
                data = pickle.load(f)
                self.model = data['model']
                self.feature_names = data['feature_names']
                self.is_trained = data['is_trained']
            return True
        return False


print("StockPredictor class defined")

## Trading Strategy

In [ ]:
# Cell 5: Trading Strategy Logic

class TradingStrategy:
    """
    Implements trading decisions based on predictions and risk management.
    """
    
    def __init__(self, predictor: StockPredictor):
        self.predictor = predictor
        
        # Strategy parameters
        self.buy_threshold = 0.02  # Buy if predicted return > 2%
        self.sell_threshold = -0.02  # Sell if predicted return < -2%
        self.max_position_pct = 0.35  # Max 35% of portfolio in one stock
        self.min_cash_reserve = 0.15  # Keep 15% cash
        self.take_profit_threshold = 0.15  # Take profit at 15% gain
    
    def generate_trading_decisions(
        self,
        portfolio: Portfolio,
        stock_data: Dict[str, Dict],
        days_remaining: int
    ) -> List[Tuple[str, str, int]]:
        """
        Generate trading decisions for the current day.
        
        Args:
            portfolio: Current portfolio state
            stock_data: Dict mapping stock name to {prices, owned}
            days_remaining: Number of days left in simulation
        
        Returns:
            List of (stock_name, action, shares) tuples
        """
        transactions = []
        
        # Calculate predictions for all stocks
        predictions = {}
        for stock_name, data in stock_data.items():
            features = calculate_technical_features(data['prices'])
            pred_return = self.predictor.predict(features)
            predictions[stock_name] = {
                'return': pred_return,
                'current_price': data['prices'][-1],
                'owned': data['owned'],
                'features': features
            }
        
        # Phase 1: Sell decisions (liquidate bad positions)
        for stock_name, pred_data in predictions.items():
            if pred_data['owned'] == 0:
                continue
            
            # Sell if:
            # 1. Predicted to decline significantly
            # 2. Hit take-profit target
            # 3. Last day (liquidate everything)
            
            should_sell = False
            
            if days_remaining == 1:
                # Last day - sell everything
                should_sell = True
            elif pred_data['return'] < self.sell_threshold:
                # Predicted decline
                should_sell = True
            else:
                # Check for take-profit
                momentum = pred_data['features'].get('momentum_5d', 0.0)
                if momentum > self.take_profit_threshold:
                    should_sell = True
            
            if should_sell:
                shares_to_sell = pred_data['owned']
                transactions.append((stock_name, 'SELL', shares_to_sell))
                # Update owned count for buy phase
                predictions[stock_name]['owned'] = 0
        
        # Phase 2: Buy decisions (allocate to promising stocks)
        if days_remaining > 1:  # Don't buy on last day
            # Sort stocks by predicted return
            buy_candidates = [
                (name, data) for name, data in predictions.items()
                if data['return'] > self.buy_threshold
            ]
            buy_candidates.sort(key=lambda x: x[1]['return'], reverse=True)
            
            # Calculate available capital for buying
            available_cash = portfolio.cash
            total_value = portfolio.total_value()
            max_cash_to_spend = available_cash * (1.0 - self.min_cash_reserve)
            
            # Allocate to top candidates
            for stock_name, pred_data in buy_candidates[:3]:  # Top 3 stocks
                if max_cash_to_spend < pred_data['current_price']:
                    break  # Not enough cash
                
                # Calculate position size
                max_position_value = total_value * self.max_position_pct
                current_value = pred_data['owned'] * pred_data['current_price']
                available_for_stock = min(
                    max_cash_to_spend,
                    max_position_value - current_value
                )
                
                if available_for_stock < pred_data['current_price']:
                    continue  # Can't afford even 1 share
                
                # Buy shares
                shares_to_buy = int(available_for_stock / pred_data['current_price'])
                if shares_to_buy > 0:
                    cost = shares_to_buy * pred_data['current_price']
                    transactions.append((stock_name, 'BUY', shares_to_buy))
                    max_cash_to_spend -= cost
        
        return transactions


print("TradingStrategy class defined")

## State Management & Persistence

In [ ]:
# Cell 6: State Persistence

class StateManager:
    """
    Manages persistent state between trading days.
    """
    
    def __init__(self, state_dir: str):
        self.state_dir = state_dir
        self.portfolio_file = os.path.join(state_dir, 'portfolio.json')
        self.model_file = os.path.join(state_dir, 'model.pkl')
        self.history_file = os.path.join(state_dir, 'price_history.json')
    
    def save_portfolio(self, portfolio: Portfolio):
        """Save portfolio state"""
        data = {
            'cash': portfolio.cash,
            'pending_cash': portfolio.pending_cash,
            'day': portfolio.day,
            'holdings': {
                name: {
                    'shares': holding.shares,
                    'price_history': holding.price_history
                }
                for name, holding in portfolio.holdings.items()
            },
            'transaction_history': portfolio.transaction_history
        }
        with open(self.portfolio_file, 'w') as f:
            json.dump(data, f, indent=2)
    
    def load_portfolio(self) -> Portfolio:
        """Load portfolio state"""
        if not os.path.exists(self.portfolio_file):
            return Portfolio()
        
        with open(self.portfolio_file, 'r') as f:
            data = json.load(f)
        
        portfolio = Portfolio(
            cash=data['cash'],
            pending_cash=data['pending_cash'],
            day=data['day']
        )
        
        for name, holding_data in data['holdings'].items():
            portfolio.holdings[name] = StockHolding(
                name=name,
                shares=holding_data['shares'],
                price_history=holding_data['price_history']
            )
        
        portfolio.transaction_history = data['transaction_history']
        return portfolio
    
    def save_price_history(self, history: Dict[str, List[float]]):
        """Save extended price history"""
        with open(self.history_file, 'w') as f:
            json.dump(history, f)
    
    def load_price_history(self) -> Dict[str, List[float]]:
        """Load extended price history"""
        if not os.path.exists(self.history_file):
            return {}
        
        with open(self.history_file, 'r') as f:
            return json.load(f)
    
    def save_model(self, predictor: StockPredictor):
        """Save ML model"""
        predictor.save(self.model_file)
    
    def load_model(self) -> StockPredictor:
        """Load ML model"""
        predictor = StockPredictor()
        predictor.load(self.model_file)
        return predictor


print("StateManager class defined")

## Main Trading Function

This is the core function that will be called each trading day with new market data.

In [ ]:
# Cell 7: Main printTransactions Function

def printTransactions(
    m: float,
    k: int,
    d: int,
    name: List[str],
    owned: List[int],
    prices: List[List[float]]
):
    """
    Main trading function called each day.
    
    Args:
        m: Money available (cash)
        k: Number of stocks
        d: Days remaining
        name: List of stock names
        owned: List of shares owned per stock
        prices: List of price histories (5 days each)
    """
    # Initialize state manager
    state_mgr = StateManager(STATE_DIR)
    
    # Load or initialize portfolio
    portfolio = state_mgr.load_portfolio()
    
    # Update portfolio with current state from input
    portfolio.start_new_day()  # Process pending cash from yesterday
    portfolio.cash = m
    
    # Update stock holdings and price history
    for i in range(k):
        stock_name = name[i]
        
        if stock_name not in portfolio.holdings:
            portfolio.holdings[stock_name] = StockHolding(stock_name, 0)
        
        portfolio.holdings[stock_name].shares = owned[i]
        portfolio.holdings[stock_name].price_history = prices[i]
    
    # Load extended price history
    price_history = state_mgr.load_price_history()
    
    # Update extended history
    for i in range(k):
        stock_name = name[i]
        if stock_name not in price_history:
            price_history[stock_name] = []
        
        # Add new prices (avoid duplicates)
        current_history = price_history[stock_name]
        for price in prices[i]:
            if not current_history or price != current_history[-1]:
                current_history.append(price)
        
        # Keep last 100 days max
        price_history[stock_name] = current_history[-100:]
    
    # Load or train model
    predictor = state_mgr.load_model()
    
    # If model not trained and we have enough data, train it
    if not predictor.is_trained and len(price_history) > 0:
        # Check if we have enough data points
        total_days = sum(len(h) for h in price_history.values())
        if total_days > 30:  # Need enough data for training
            try:
                train_df = create_training_dataset(price_history)
                if len(train_df) > 20:
                    feature_cols = [c for c in train_df.columns 
                                   if c not in ['stock', 'target_return', 'target_direction']]
                    X = train_df[feature_cols]
                    y = train_df['target_return']
                    predictor.train(X, y)
                    print("# Model trained successfully", flush=True)
            except Exception as e:
                print(f"# Training error: {e}", flush=True)
    
    # Create trading strategy
    strategy = TradingStrategy(predictor)
    
    # Prepare stock data for strategy
    stock_data = {
        name[i]: {
            'prices': prices[i],
            'owned': owned[i]
        }
        for i in range(k)
    }
    
    # Generate trading decisions
    transactions = strategy.generate_trading_decisions(
        portfolio, stock_data, d
    )
    
    # Execute transactions and output
    print(len(transactions))
    
    for stock_name, action, shares in transactions:
        current_price = stock_data[stock_name]['prices'][-1]
        
        if action == 'BUY' and portfolio.can_buy(stock_name, shares, current_price):
            portfolio.execute_buy(stock_name, shares, current_price)
            print(f"{stock_name} BUY {shares}")
        elif action == 'SELL' and portfolio.can_sell(stock_name, shares):
            portfolio.execute_sell(stock_name, shares, current_price)
            print(f"{stock_name} SELL {shares}")
    
    # Save state
    state_mgr.save_portfolio(portfolio)
    state_mgr.save_price_history(price_history)
    state_mgr.save_model(predictor)
    
    # Debug output (commented out for submission)
    # print(f"# Day {portfolio.day}, Cash: ${portfolio.cash:.2f}, "
    #       f"Portfolio Value: ${portfolio.total_value():.2f}, "
    #       f"Score: {portfolio.get_score():.2f}", flush=True)


print("printTransactions function defined")

## Testing & Validation

In [ ]:
# Cell 8: Test with Sample Input

def test_sample_input():
    """
    Test with the sample input from problem description.
    """
    print("=" * 60)
    print("Testing with sample input")
    print("=" * 60)
    
    # Sample input from problem
    m = 90.0
    k = 2
    d = 400
    name = ['iStreet', 'HR']
    owned = [10, 0]
    prices = [
        [4.54, 5.53, 6.56, 5.54, 7.60],
        [30.54, 27.53, 24.42, 20.11, 17.50]
    ]
    
    print("\nInput:")
    print(f"Money: ${m}")
    print(f"Stocks: {k}")
    print(f"Days remaining: {d}")
    for i in range(k):
        print(f"  {name[i]}: owned={owned[i]}, prices={prices[i]}")
    
    print("\nOutput:")
    printTransactions(m, k, d, name, owned, prices)
    
    print("\n" + "=" * 60)
    print("Expected output format (from problem):")
    print("2")
    print("iStreet SELL 10")
    print("HR BUY 5")
    print("=" * 60)


# Run test
test_sample_input()

## Performance Analysis

In [ ]:
# Cell 9: Analyze Performance

def analyze_performance():
    """
    Analyze trading performance from saved state.
    """
    state_mgr = StateManager(STATE_DIR)
    portfolio = state_mgr.load_portfolio()
    
    print("\n" + "=" * 60)
    print("PERFORMANCE SUMMARY")
    print("=" * 60)
    
    print(f"\nCurrent Portfolio State:")
    print(f"  Trading Day: {portfolio.day}")
    print(f"  Cash: ${portfolio.cash:.2f}")
    print(f"  Pending Cash: ${portfolio.pending_cash:.2f}")
    
    print(f"\nHoldings:")
    for name, holding in portfolio.holdings.items():
        if holding.shares > 0:
            value = holding.current_value()
            print(f"  {name}: {holding.shares} shares @ ${holding.current_price():.2f} = ${value:.2f}")
    
    total_value = portfolio.total_value()
    score = portfolio.get_score()
    
    print(f"\nPortfolio Metrics:")
    print(f"  Total Value: ${total_value:.2f}")
    print(f"  Return: {(total_value / 100.0 - 1.0) * 100:.2f}%")
    print(f"  Score: {score:.4f}")
    
    if portfolio.transaction_history:
        print(f"\nRecent Transactions (last 10):")
        for txn in portfolio.transaction_history[-10:]:
            print(f"  Day {txn['day']}: {txn['action']} {txn['shares']} {txn['stock']} "
                  f"@ ${txn['price']:.2f} (${txn['value']:.2f})")
    
    print("\n" + "=" * 60)


# Analyze current performance
try:
    analyze_performance()
except Exception as e:
    print(f"Performance analysis not available yet: {e}")

## Simulation Mode

Optional: Simulate multiple days of trading with synthetic data.

In [ ]:
# Cell 10: Multi-Day Simulation (Optional)

def simulate_trading_days(num_days: int = 10):
    """
    Simulate multiple trading days with synthetic price movements.
    Useful for testing the complete system.
    """
    print(f"\nSimulating {num_days} trading days...\n")
    
    # Initialize with sample stocks
    stocks = ['AAPL', 'GOOGL', 'MSFT']
    base_prices = [150.0, 2800.0, 350.0]
    
    # Generate price history
    price_histories = {}
    for stock, base in zip(stocks, base_prices):
        prices = [base]
        for _ in range(20):
            # Random walk with slight upward drift
            change = np.random.normal(0.002, 0.02)
            prices.append(prices[-1] * (1 + change))
        price_histories[stock] = prices
    
    # Simulate trading
    m = 100.0  # Starting cash
    owned = [0, 0, 0]  # No initial holdings
    
    for day in range(num_days):
        print(f"\n--- Day {day + 1} ---")
        
        # Get current 5-day windows
        current_prices = []
        for stock in stocks:
            window_start = min(day, len(price_histories[stock]) - 5)
            window = price_histories[stock][window_start:window_start + 5]
            current_prices.append(window)
        
        # Call trading function
        printTransactions(
            m=m,
            k=len(stocks),
            d=num_days - day,
            name=stocks,
            owned=owned,
            prices=current_prices
        )
        
        # Update owned shares (in real game, this comes from next input)
        # For simulation, we'd need to parse the output
        # This is simplified
    
    print("\nSimulation complete!")
    analyze_performance()


# Uncomment to run simulation
# simulate_trading_days(10)

## Reset State

Use this to clear all saved state and start fresh.

In [ ]:
# Cell 11: Reset State

def reset_state():
    """
    Clear all saved state files.
    Use this to start a fresh trading session.
    """
    import shutil
    
    if os.path.exists(STATE_DIR):
        shutil.rmtree(STATE_DIR)
        os.makedirs(STATE_DIR)
        print(f"✓ State directory cleared: {STATE_DIR}")
    else:
        print("No state to clear")


# Uncomment to reset state
# reset_state()

## Summary

This notebook implements a complete algorithmic trading system for stock price prediction:

### Components
1. **Portfolio Management**: Track cash, holdings, and transactions
2. **Feature Engineering**: Technical indicators from price history
3. **ML Model**: XGBoost predictor for price movements
4. **Trading Strategy**: Buy/sell decisions with risk management
5. **State Persistence**: Save model and portfolio between days

### Key Features
- Predicts next-day returns using technical indicators
- Risk management: position sizing, stop losses, take profits
- Learns patterns from historical data
- Maintains state across trading days
- Optimizes for scoring function: 5 × ln(money)

### Usage
```python
# Main function called each day
printTransactions(m, k, d, name, owned, prices)
```

### Next Steps
1. Deploy to Snowflake workspace
2. Connect to actual stock data tables
3. Register model in Model Registry
4. Set up monitoring with ML Observability

In [ ]:
printTransactions(
    m=90.0,
    k=2,
    d=400,
    name=['iStreet', 'HR'],
    owned=[10, 0],
    prices=[
        [4.54, 5.53, 6.56, 5.54, 7.60],
        [30.54, 27.53, 24.42, 20.11, 17.50]
    ]
)